# LeetCode 3936 — Minimum Swaps to Move Zeros to End bằng Python

Link đề: https://leetcode.com/problems/minimum-swaps-to-move-zeros-to-end/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def minimumSwaps(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **hoán đổi giá trị ở hai chỉ số bất kỳ**; cần số thao tác **ít nhất** để mọi số `0` nằm ở cuối mảng.
2. Nhận ra **bước đột phá đếm**: đáp án bằng số số `0` đang nằm trong `n - z` ô đầu (`z` là tổng số số `0`).
3. Biết dùng `list.count`, lát cắt `nums[:k]`, hai con trỏ trái/phải và mô phỏng hoán đổi `a[i], a[j] = a[j], a[i]`.
4. Viết 4 cách (BFS vét cạn cho mảng nhỏ, đếm lát cắt, hai con trỏ, mô phỏng) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: đếm hết số `0`, nhầm với bài dồn số `0` bằng hoán đổi **kề nhau**, dùng sai độ dài lát cắt (`nums[:z]`).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trong một thao tác bạn được chọn **hai chỉ số khác nhau** `i != j` và **hoán đổi** `nums[i]` với `nums[j]`. Hãy trả về **số thao tác ít nhất** để mọi số `0` nằm ở cuối mảng (các số khác `0` đứng trước, thứ tự giữa chúng **không quan trọng**).

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[0,1,0,3,12]` | `2` | Đổi chỉ số 0 và 3 -> `[3,1,0,0,12]`; đổi chỉ số 2 và 4 -> `[3,1,12,0,0]` |
| 2 | `[0,1,0,2]` | `1` | Đổi chỉ số 0 và 3 -> `[2,1,0,0]` |
| 3 | `[1,2,0]` | `0` | Số `0` đã ở cuối |

Constraints: `1 <= len(nums) <= 100`; `0 <= nums[i] <= 100`.

Điểm khác với bài kinh điển "Move Zeroes" (283): ở đây ta **đếm số lần hoán đổi tối thiểu**, và mỗi lần hoán đổi được chọn **hai vị trí bất kỳ** (không cần kề nhau).

In [1]:
examples = [([0, 1, 0, 3, 12], 2), ([0, 1, 0, 2], 1), ([1, 2, 0], 0)]
for nums, expected in examples:
    z = nums.count(0)
    n = len(nums)
    print(f"nums={nums}  n={n}  z={z}  ô đầu (n-z={n - z}) = {nums[:n - z]}  -> expected={expected}")

nums=[0, 1, 0, 3, 12]  n=5  z=2  ô đầu (n-z=3) = [0, 1, 0]  -> expected=2
nums=[0, 1, 0, 2]  n=4  z=2  ô đầu (n-z=2) = [0, 1]  -> expected=1
nums=[1, 2, 0]  n=3  z=1  ô đầu (n-z=2) = [1, 2]  -> expected=0


## 3. Kiến thức nền

### 3.1. `list.count(x)` đếm số lần xuất hiện

`nums.count(0)` cho số số `0` trong mảng. Độ phức tạp `O(n)`.

In [2]:
nums = [0, 1, 0, 3, 12]
print(nums.count(0), nums.count(5), len(nums) - nums.count(0))

2 0 3


### 3.2. Lát cắt `nums[a:b]`

`nums[:k]` là `k` phần tử đầu (chỉ số `0..k-1`); `nums[k:]` là phần còn lại. Với `k = 0`, `nums[:0] = []`. Lát cắt tạo **list mới**, không sửa `nums`.

In [3]:
nums = [0, 1, 0, 3, 12]
k = 3
print(nums[:k], nums[k:], nums[:0], nums[:k].count(0))

[0, 1, 0] [3, 12] [] 2


### 3.3. Hoán đổi hai phần tử

Python cho phép hoán đổi không cần biến tạm: `a[i], a[j] = a[j], a[i]`. Vế phải được tính xong (thành tuple) trước khi gán.

In [4]:
a = [0, 1, 0, 3, 12]
a[0], a[3] = a[3], a[0]
print(a)

[3, 1, 0, 0, 12]


### 3.4. Hai con trỏ ngược chiều

Đặt `l = 0` ở đầu, `r = n - 1` ở cuối, di chuyển vào trong cho tới khi gặp nhau (`l < r`). Mỗi con trỏ "bỏ qua" những phần tử đã đúng chỗ và dừng ở phần tử sai chỗ.

In [5]:
a = [0, 1, 0, 3, 12]
l, r = 0, len(a) - 1
while l < r:
    print(f"l={l}, r={r}, a[l]={a[l]}, a[r]={a[r]}")
    l += 1
    r -= 1

l=0, r=4, a[l]=0, a[r]=12
l=1, r=3, a[l]=1, a[r]=3


### 3.5. Tìm đường ngắn nhất bằng BFS trên "trạng thái"

Khi cần "số bước ít nhất" và số trạng thái nhỏ, ta coi **mỗi mảng là một đỉnh**, mỗi hoán đổi là một cạnh, rồi BFS từ mảng ban đầu tới khi gặp một mảng "đã xong". Số trạng thái tối đa `n!` nên chỉ dùng cho `n <= 7` (làm **trọng tài** kiểm tra đáp án, không dùng để nộp).

In [6]:
from collections import deque

def done(a):
    seen_zero = False
    for x in a:
        if x == 0:
            seen_zero = True
        elif seen_zero:
            return False
    return True

print(done((1, 2, 0, 0)), done((0, 1)), done((1, 0, 2)))
q = deque([((0, 1), 0)])
print(q.popleft())

True False False
((0, 1), 0)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `z` là **tổng số** số `0`. Khi xong, `z` số `0` chiếm `z` ô **cuối**, các số khác chiếm `n - z` ô **đầu**. Vậy ô nào "sai chỗ"?

- **Số `0` nằm trong `n - z` ô đầu** là sai chỗ (đáng lẽ chỗ đó phải là số khác `0`);
- **Số khác `0` nằm trong `z` ô cuối** cũng sai chỗ.

Hai nhóm này **có số lượng bằng nhau** (vì tổng số `0` là `z`: nếu `m` số `0` nằm ở vùng đầu thì còn `z - m` số `0` ở vùng cuối `z` ô, nên có `m` ô vùng cuối bị số khác `0` chiếm). Mỗi lần hoán đổi một số `0` sai chỗ với một số khác `0` sai chỗ sửa được cả hai.

Tính tay `[0,1,0,3,12]`: `n = 5`, `z = 2`, vùng đầu là 3 ô `[0,1,0]` chứa 2 số `0` -> cần `2` hoán đổi.
Tính tay `[0,1,0,2]`: `n = 4`, `z = 2`, vùng đầu 2 ô `[0,1]` chứa 1 số `0` -> `1` hoán đổi.

In [7]:
def explain(nums):
    n, z = len(nums), nums.count(0)
    head, tail = nums[:n - z], nums[n - z:]
    print(f"{nums}: vùng đầu={head} (có {head.count(0)} số 0), vùng cuối={tail} (có {len(tail) - tail.count(0)} số khác 0)")

for a in ([0, 1, 0, 3, 12], [0, 1, 0, 2], [1, 2, 0], [0, 0, 5], [5, 0, 0]):
    explain(a)

[0, 1, 0, 3, 12]: vùng đầu=[0, 1, 0] (có 2 số 0), vùng cuối=[3, 12] (có 2 số khác 0)
[0, 1, 0, 2]: vùng đầu=[0, 1] (có 1 số 0), vùng cuối=[0, 2] (có 1 số khác 0)
[1, 2, 0]: vùng đầu=[1, 2] (có 0 số 0), vùng cuối=[0] (có 0 số khác 0)
[0, 0, 5]: vùng đầu=[0] (có 1 số 0), vùng cuối=[0, 5] (có 1 số khác 0)
[5, 0, 0]: vùng đầu=[5] (có 0 số 0), vùng cuối=[0, 0] (có 0 số khác 0)


## 5. Cách 1 — Brute Force / baseline: BFS vét cạn trên trạng thái

Làm **đúng nghĩa đen** của đề: thử mọi hoán đổi `(i, j)`, BFS theo số bước, dừng khi mảng có mọi số `0` ở cuối. Đúng chắc chắn (BFS cho đường ngắn nhất) nhưng số trạng thái lên tới `n!`, nên **chỉ chạy được với `n` rất nhỏ** (khoảng `n <= 7`). Với `n = 100` (constraints thật) cách này vô dụng: đó là "nút thắt" cần đột phá bằng công thức đếm.

In [8]:
from collections import deque

class SolutionBFS:
    def minimumSwaps(self, nums):
        start = tuple(nums)
        n = len(start)
        def done(a):
            seen_zero = False
            for x in a:
                if x == 0:
                    seen_zero = True
                elif seen_zero:
                    return False
            return True
        if done(start):
            return 0
        seen = {start}
        q = deque([(start, 0)])
        while q:
            a, d = q.popleft()
            for i in range(n):
                for j in range(i + 1, n):
                    if a[i] == a[j]:
                        continue                      # đổi hai giá trị bằng nhau không đổi mảng
                    b = list(a)
                    b[i], b[j] = b[j], b[i]
                    t = tuple(b)
                    if t in seen:
                        continue
                    if done(t):
                        return d + 1
                    seen.add(t)
                    q.append((t, d + 1))
        return 0

print(SolutionBFS().minimumSwaps([0, 1, 0, 3, 12]), SolutionBFS().minimumSwaps([0, 1, 0, 2]), SolutionBFS().minimumSwaps([1, 2, 0]))

2 1 0


**Độ phức tạp:** Time `O(n! * n^2)` trong trường hợp xấu, Space `O(n!)`. Chỉ dùng làm trọng tài cho `n <= 7`.

## 6. Ý tưởng thuật toán / bước đột phá

**Khẳng định:** đáp án = **số số `0` nằm trong `n - z` ô đầu** (với `z = nums.count(0)`).

**Chứng minh hai chiều (cận dưới và cận trên):**

- *Cận trên (đủ):* gọi `m` là số số `0` ở vùng đầu. Có đúng `m` số khác `0` ở vùng cuối. Ghép từng cặp (một số `0` ở vùng đầu, một số khác `0` ở vùng cuối) rồi hoán đổi: mỗi cặp đúng 1 hoán đổi, sau `m` hoán đổi mọi số `0` đã ở vùng cuối. Vậy `m` hoán đổi là đủ.
- *Cận dưới (cần):* trước khi xong có `2m` ô sai chỗ (`m` số `0` ở vùng đầu + `m` số khác `0` ở vùng cuối). Một hoán đổi chỉ đụng tới **2 ô** nên sửa được **tối đa 2 ô sai chỗ**. Vậy cần ít nhất `2m / 2 = m` hoán đổi.

**Quy tắc "nếu... thì...":** nếu `0` đứng ở vùng đầu `[0, n - z)` thì nó "tốn" đúng một hoán đổi; mọi vị trí khác không tốn gì.

## 7. Các cách tiếp theo

### Cách 2 — Đếm số `0` trong lát cắt đầu

Hai dòng: `z = nums.count(0)`, trả `nums[:len(nums) - z].count(0)`.

In [9]:
class SolutionCount:
    def minimumSwaps(self, nums):
        z = nums.count(0)
        return nums[:len(nums) - z].count(0)

print(SolutionCount().minimumSwaps([0, 1, 0, 3, 12]), SolutionCount().minimumSwaps([0, 1, 0, 2]), SolutionCount().minimumSwaps([1, 2, 0]))

2 1 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho lát cắt (có thể `O(1)` nếu đếm bằng chỉ số).

### Cách 3 — Hai con trỏ trái/phải (mô phỏng thật)

`l` đi từ trái tìm số `0` (sai chỗ ở vùng đầu), `r` đi từ phải tìm số khác `0` (sai chỗ ở vùng cuối). Gặp đủ cặp thì đếm một hoán đổi (và thực sự hoán đổi trên bản sao để thấy mảng thay đổi).

In [10]:
class SolutionTwoPointers:
    def minimumSwaps(self, nums):
        a = list(nums)                                # không sửa đầu vào
        l, r = 0, len(a) - 1
        swaps = 0
        while True:
            while l < r and a[l] != 0:               # bỏ qua số khác 0 ở bên trái
                l += 1
            while l < r and a[r] == 0:               # bỏ qua số 0 ở bên phải
                r -= 1
            if l >= r:
                break
            a[l], a[r] = a[r], a[l]
            swaps += 1
            l += 1
            r -= 1
        return swaps

print(SolutionTwoPointers().minimumSwaps([0, 1, 0, 3, 12]), SolutionTwoPointers().minimumSwaps([0, 1, 0, 2]), SolutionTwoPointers().minimumSwaps([1, 2, 0]))

2 1 0


**Độ phức tạp:** Time `O(n)` (mỗi con trỏ đi tối đa `n` bước), Space `O(n)` cho bản sao `a` (bỏ bản sao nếu được phép sửa đầu vào: `O(1)`).

### Cách 4 — Đếm số khác `0` ở vùng cuối (cách nhìn đối ngẫu)

Số `0` sai chỗ ở vùng đầu **bằng** số số khác `0` sai chỗ ở vùng cuối, nên có thể đếm bên nào cũng được: `sum(1 for x in nums[n - z:] if x != 0)`.

In [11]:
class SolutionTail:
    def minimumSwaps(self, nums):
        n = len(nums)
        z = nums.count(0)
        return sum(1 for x in nums[n - z:] if x != 0)

print(SolutionTail().minimumSwaps([0, 1, 0, 3, 12]), SolutionTail().minimumSwaps([0, 1, 0, 2]), SolutionTail().minimumSwaps([1, 2, 0]))

2 1 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)` nếu dùng chỉ số thay lát cắt (ở đây `O(z)` do lát cắt).

## 8. TRACE từng bước

Trace Cách 3 với `nums = [0,1,0,3,12]` (`n = 5`):

| bước | `l` | `r` | `a` | hành động | `swaps` |
|---|---|---|---|---|---|
| đầu | 0 | 4 | `[0,1,0,3,12]` | `a[0] = 0` đứng yên, `a[4] = 12 != 0` đứng yên | 0 |
| 1 | 0 | 4 | `[0,1,0,3,12]` | đổi `a[0]` với `a[4]` | 1 |
| 2 | 1 | 3 | `[12,1,0,3,0]` | `l` tiến tới 2 (a[2]=0), `r` ở 3 (a[3]=3 != 0) -> đổi | 2 |
| kết | 3 | 2 | `[12,1,3,0,0]` | `l >= r` thoát | 2 |

(Cách hoán đổi này khác ví dụ của đề nhưng cùng số lần: `2`.)

In [12]:
def trace_two_pointers(nums):
    a = list(nums)
    l, r, swaps = 0, len(a) - 1, 0
    print("bắt đầu:", a)
    while True:
        while l < r and a[l] != 0:
            l += 1
        while l < r and a[r] == 0:
            r -= 1
        print(f"  l={l}, r={r}, a={a}")
        if l >= r:
            break
        a[l], a[r] = a[r], a[l]
        swaps += 1
        print(f"  -> đổi a[{l}] với a[{r}]: {a}, swaps={swaps}")
        l += 1
        r -= 1
    print("kết quả:", swaps, a)
    return swaps

assert trace_two_pointers([0, 1, 0, 3, 12]) == 2
assert trace_two_pointers([1, 2, 0]) == 0

bắt đầu: [0, 1, 0, 3, 12]
  l=0, r=4, a=[0, 1, 0, 3, 12]
  -> đổi a[0] với a[4]: [12, 1, 0, 3, 0], swaps=1
  l=2, r=3, a=[12, 1, 0, 3, 0]
  -> đổi a[2] với a[3]: [12, 1, 3, 0, 0], swaps=2
  l=3, r=2, a=[12, 1, 3, 0, 0]
kết quả: 2 [12, 1, 3, 0, 0]
bắt đầu: [1, 2, 0]
  l=2, r=2, a=[1, 2, 0]
kết quả: 0 [1, 2, 0]


In [13]:
def trace_count(nums):
    n, z = len(nums), nums.count(0)
    print(f"nums={nums}, n={n}, z={z}, vùng đầu nums[:{n - z}]={nums[:n - z]}")
    for i in range(n - z):
        print(f"  ô {i}: giá trị {nums[i]} ->", "SAI CHỖ (số 0)" if nums[i] == 0 else "đúng chỗ")
    return nums[:n - z].count(0)

assert trace_count([0, 1, 0, 2]) == 1

nums=[0, 1, 0, 2], n=4, z=2, vùng đầu nums[:2]=[0, 1]
  ô 0: giá trị 0 -> SAI CHỖ (số 0)
  ô 1: giá trị 1 -> đúng chỗ


## 9. Code nộp LeetCode

Dùng Cách 2 (đếm), `O(n)`.

In [14]:
from typing import List, Optional


class Solution:
    def minimumSwaps(self, nums: List[int]) -> int:
        z = nums.count(0)
        return nums[:len(nums) - z].count(0)


print(Solution().minimumSwaps([0, 1, 0, 3, 12]), Solution().minimumSwaps([0, 1, 0, 2]), Solution().minimumSwaps([1, 2, 0]))

2 1 0


## 10. Vì sao thuật toán đúng

1. **Trạng thái đích.** Mảng "xong" khi mọi số `0` đứng cuối, nghĩa là `z` ô cuối đều `0` và `n - z` ô đầu đều khác `0`.
2. **Ô sai chỗ.** Gọi `m` = số số `0` ở vùng đầu `[0, n - z)`. Vùng cuối có `z` ô, chứa `z - m` số `0`, nên chứa `m` số khác `0`. Tổng ô sai chỗ là `2m`.
3. **Cận dưới.** Hoán đổi `(i, j)` chỉ thay đổi hai ô, nên số ô sai chỗ giảm **không quá 2** mỗi lần. Từ `2m` ô sai về `0` cần `>= m` lần.
4. **Cận trên.** Mỗi lần hoán đổi một số `0` ở vùng đầu với một số khác `0` ở vùng cuối giảm đúng 2 ô sai chỗ -> `m` lần là đủ.
5. **Kết luận.** `min = m`. Mọi cách đều tính cùng đại lượng `m`: Cách 2 đếm số `0` ở vùng đầu, Cách 4 đếm số khác `0` ở vùng cuối (bằng nhau), Cách 3 thực sự thực hiện `m` hoán đổi, Cách 1 (BFS) tìm trực tiếp khoảng cách nhỏ nhất.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `z` = số số `0`.

- Cách 1 (BFS): Time `O(n! * n^2)`, Space `O(n!)`; chỉ dùng cho `n <= 7`.
- Cách 2 (nộp): `count` toàn bộ `O(n)` + lát cắt và đếm `O(n - z)`; Time `O(n)`, Space `O(n)` (có thể `O(1)` nếu duyệt bằng chỉ số).
- Cách 3: Time `O(n)`; Space `O(n)` cho bản sao.
- Cách 4: Time `O(n)`, Space `O(z)` do lát cắt.

Cận dưới thời gian: phải xem mọi phần tử ít nhất một lần nên `O(n)` là tối ưu về bậc.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionCount(), SolutionTwoPointers(), SolutionTail(), Solution()]

fixed = [
    ([0, 1, 0, 3, 12], 2), ([0, 1, 0, 2], 1), ([1, 2, 0], 0),
    ([0], 0), ([5], 0), ([0, 0], 0), ([0, 5], 1), ([5, 0], 0),
    ([0, 0, 5], 1), ([0, 0, 5, 7], 2), ([5, 7, 0, 0], 0), ([0, 1, 0, 2, 0], 1),
    ([3, 0, 4, 0, 5, 0], 1),
]
for nums, expected in fixed:
    for sol in solutions + [SolutionBFS()]:
        arg = list(nums)
        assert sol.minimumSwaps(arg) == expected, (type(sol).__name__, nums)
        assert arg == nums, "đầu vào bị sửa: " + type(sol).__name__
print("test cố định OK:", len(fixed), "ca")

random.seed(3936)
for _ in range(300):
    n = random.randint(1, 7)                       # đủ nhỏ để BFS làm trọng tài
    nums = [random.choice([0, 0, random.randint(1, 100)]) for _ in range(n)]
    expected = SolutionBFS().minimumSwaps(nums)
    for sol in solutions:
        assert sol.minimumSwaps(list(nums)) == expected, (type(sol).__name__, nums)
print("đối chiếu với BFS OK (n <= 7)")

for _ in range(400):
    n = random.randint(1, 100)                     # đúng constraints, so các cách nhanh với nhau
    p = random.random()
    nums = [0 if random.random() < p else random.randint(1, 100) for _ in range(n)]
    outs = {sol.minimumSwaps(list(nums)) for sol in solutions}
    assert len(outs) == 1, nums
print("test ngẫu nhiên n <= 100 OK")

test cố định OK: 13 ca
đối chiếu với BFS OK (n <= 7)
test ngẫu nhiên n <= 100 OK


### Test thuộc tính (property test)

(1) Đáp án nằm trong `[0, min(z, n - z)]`. (2) Sau khi **thực sự** áp dụng `m` hoán đổi (ghép số `0` ở vùng đầu với số khác `0` ở vùng cuối) thì mảng "xong". (3) Đáp án không đổi khi thay mọi số khác `0` bằng số khác `0` bất kỳ.

In [16]:
def apply_greedy(nums):
    a = list(nums); n = len(a); z = a.count(0)
    bad_head = [i for i in range(n - z) if a[i] == 0]
    bad_tail = [j for j in range(n - z, n) if a[j] != 0]
    assert len(bad_head) == len(bad_tail)
    for i, j in zip(bad_head, bad_tail):
        a[i], a[j] = a[j], a[i]
    return a, len(bad_head)

random.seed(39360)
for _ in range(500):
    n = random.randint(1, 40)
    nums = [random.choice([0, 0, random.randint(1, 100)]) for _ in range(n)]
    ans = Solution().minimumSwaps(nums)
    z = nums.count(0)
    assert 0 <= ans <= min(z, n - z)
    final, used = apply_greedy(nums)
    assert used == ans and final == sorted(final, key=lambda x: x == 0)
    relabeled = [0 if x == 0 else random.randint(1, 100) for x in nums]
    assert Solution().minimumSwaps(relabeled) == ans
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả về tổng số số `0`

Chỉ những số `0` **không ở cuối** mới tốn công; số `0` đã nằm đúng vùng cuối thì không cần động. Trả `nums.count(0)` sai khi mảng đã đúng như `[1,2,0]` (đáp án `0`, nhưng `count` cho `1`).

In [17]:
def wrong_all_zeros(nums):
    return nums.count(0)                               # SAI: đếm cả số 0 đã đúng chỗ

print(wrong_all_zeros([1, 2, 0]), "!=", Solution().minimumSwaps([1, 2, 0]))
assert wrong_all_zeros([1, 2, 0]) == 1 and Solution().minimumSwaps([1, 2, 0]) == 0
assert wrong_all_zeros([0, 1, 0, 2, 0]) == 3 and Solution().minimumSwaps([0, 1, 0, 2, 0]) == 1

1 != 0


### Lỗi 2: dùng sai độ dài vùng đầu (`nums[:z]` thay vì `nums[:n - z]`)

Vùng đầu có **`n - z`** ô (chỗ dành cho số khác `0`), không phải `z` ô.

In [18]:
def wrong_slice(nums):
    z = nums.count(0)
    return nums[:z].count(0)                           # SAI: phải nums[:len(nums) - z]

print(wrong_slice([0, 1, 0, 3, 12]), "!=", Solution().minimumSwaps([0, 1, 0, 3, 12]))
assert wrong_slice([0, 1, 0, 3, 12]) == 1 and Solution().minimumSwaps([0, 1, 0, 3, 12]) == 2

1 != 2


### Lỗi 3: nhầm với hoán đổi **kề nhau** (đếm nghịch thế)

Bài "Move Zeroes" kinh điển đổi hai phần tử **liền kề**. Khi đó số bước = số cặp (số `0` trước số khác `0`), lớn hơn nhiều. Bài này cho đổi **bất kỳ hai chỉ số**, nên không được đếm như vậy.

In [19]:
def wrong_adjacent_swaps(nums):
    zeros_before, total = 0, 0
    for x in nums:
        if x == 0:
            zeros_before += 1
        else:
            total += zeros_before                      # SAI cho bài này: đây là số hoán đổi KỀ NHAU
    return total

print(wrong_adjacent_swaps([0, 1, 0, 3, 12]), "!=", Solution().minimumSwaps([0, 1, 0, 3, 12]))
assert wrong_adjacent_swaps([0, 1, 0, 3, 12]) == 5 and Solution().minimumSwaps([0, 1, 0, 3, 12]) == 2

5 != 2


### Lỗi 4: lấy "số số `0` không phải ở cuối mảng" (chỉ trừ chuỗi `0` cuối cùng)

Cách trừ số `0` ở **đuôi liên tiếp** sai: `[0,1,0,2,0]` có `z = 3`, đuôi liên tiếp chỉ `1` số `0`, còn 2; nhưng đúng chỉ cần `1` hoán đổi (đổi `0` đầu với `2`).

In [20]:
def wrong_trailing(nums):
    z = nums.count(0)
    trailing = 0
    for x in reversed(nums):
        if x != 0:
            break
        trailing += 1
    return z - trailing                                # SAI: vẫn đếm 0 ở "giữa" nhiều hơn cần

print(wrong_trailing([0, 1, 0, 2, 0]), "!=", Solution().minimumSwaps([0, 1, 0, 2, 0]))
assert wrong_trailing([0, 1, 0, 2, 0]) == 2 and Solution().minimumSwaps([0, 1, 0, 2, 0]) == 1

2 != 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số hoán đổi **bất kỳ cặp** tối thiểu để sắp xếp một mảng nhị phân/phân vùng" -> đếm **phần tử sai chỗ** rồi chia cho 2 (mỗi hoán đổi sửa 2 ô), hoặc đếm một phía vì hai phía bằng nhau.
- Hãy tự hỏi: "đích cuối là gì?" (ở đây là `n - z` ô đầu khác 0, `z` ô cuối bằng 0) rồi đếm khác biệt so với hiện tại.
- Dùng BFS vét cạn với mảng nhỏ làm trọng tài để kiểm công thức đoán: kỹ thuật rất hữu ích cho bài "số thao tác tối thiểu".
- Phân biệt: hoán đổi **kề nhau** (đếm nghịch thế) và hoán đổi **bất kỳ** (đếm vị trí sai).
- Với `n <= 100` mọi cách `O(n)` hay `O(n^2)` đều qua; nhưng luôn nên viết cách ngắn, dễ chứng minh.

## 15. Biến thể của bài

1. **Move Zeroes** (283): dồn số `0` về cuối nhưng giữ thứ tự tương đối, **tại chỗ**.
2. Đếm số hoán đổi **kề nhau** tối thiểu để dồn số `0` về cuối: `sum` số `0` đứng trước từng số khác `0`.
3. **Minimum Swaps to Group All 1's Together** (1151, Premium) / **II** (2134): gom các số `1`.
4. **Sort Colors** (75): ba loại giá trị `0, 1, 2`.
5. Dồn số `0` về **đầu** thay vì cuối: đối xứng, đếm số `0` ở `z`... vùng cuối `n - z` ô.
6. Mảng nhị phân `0/1` thành dạng `0..01..1`: đáp án = số số `1` trong `z` ô đầu (`z` = số `0`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | thử mọi hoán đổi, BFS | `O(n! n^2)` | `O(n!)` | Chỉ `n <= 7`, làm trọng tài kiểm tra |
| 2. Đếm lát cắt | số `0` trong `n - z` ô đầu | `O(n)` | `O(n)` | **Nộp bài**, ngắn nhất |
| 3. Hai con trỏ | thực hiện hoán đổi thật | `O(n)` | `O(n)` (bản sao) | Khi cần *in ra* chuỗi hoán đổi |
| 4. Đếm vùng cuối | số khác `0` trong `z` ô cuối | `O(n)` | `O(z)` | Kiểm chứng tính đối ngẫu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minimumSwaps([0, 0, 1, 1])` và `Solution().minimumSwaps([1, 0, 2, 0, 3])` ra gì?

**Bài 2 (tính tay):** với `nums = [0, 2, 0, 0, 5, 0, 7]`, tính `n`, `z`, vùng đầu, đáp án.

**Bài 3 (điền TODO):** hoàn thành `min_swaps_zeros_front(nums)`: số hoán đổi tối thiểu để mọi số `0` ở **đầu** mảng.

**Bài 4 (tự viết lại):** viết `swap_plan(nums)` trả về danh sách các cặp chỉ số `(i, j)` cần hoán đổi (độ dài bằng đáp án).

**Bài 5 (mini challenge):** viết `min_swaps_binary_sorted(bits)`: mảng chỉ gồm `0/1`, số hoán đổi bất kỳ tối thiểu để thành dạng không giảm `0..01..1`.

In [21]:
# Bài 1: đoán rồi chạy
print(Solution().minimumSwaps([0, 0, 1, 1]), Solution().minimumSwaps([1, 0, 2, 0, 3]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def min_swaps_zeros_front_todo(nums):
    # TODO: z = số số 0; vùng cuối có n - z ô; đếm số 0 ... hay số khác 0 ở vùng nào?
    return 0

2 1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: xác định `z` và vùng đầu cho từng mảng. Bài 2: `n = 7`, đếm `z`.

**Mức 2 (vừa):** Bài 3: nếu số `0` phải ở đầu, vùng `z` ô đầu phải toàn `0`, nên đếm số khác `0` trong `nums[:z]`. Bài 4: ghép hai danh sách chỉ số sai chỗ bằng `zip`. Bài 5: đây là bài 3 cho `0/1`: đích là `z` ô đầu bằng `0`.

**Mức 3 (gần đáp án):** Bài 1: `[0,0,1,1] -> 2`, `[1,0,2,0,3] -> 1`. Bài 2: `n = 7`, `z = 4`, vùng đầu 3 ô `[0,2,0]` chứa 2 số `0` -> `2`. Bài 3: `[1,0,0] -> 1`. Bài 5: `[1,0,1,0] -> 1`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
print(Solution().minimumSwaps([0, 0, 1, 1]), Solution().minimumSwaps([1, 0, 2, 0, 3]))
assert Solution().minimumSwaps([0, 0, 1, 1]) == 2 and Solution().minimumSwaps([1, 0, 2, 0, 3]) == 1
nums2 = [0, 2, 0, 0, 5, 0, 7]
assert len(nums2) == 7 and nums2.count(0) == 4 and nums2[:3] == [0, 2, 0] and Solution().minimumSwaps(nums2) == 2

# Bài 3
def min_swaps_zeros_front(nums):
    z = nums.count(0)
    return sum(1 for x in nums[:z] if x != 0)

assert min_swaps_zeros_front([1, 0, 0]) == 1 and min_swaps_zeros_front([0, 0, 1]) == 0 and min_swaps_zeros_front([1, 2]) == 0

# Bài 4
def swap_plan(nums):
    n, z = len(nums), nums.count(0)
    head = [i for i in range(n - z) if nums[i] == 0]
    tail = [j for j in range(n - z, n) if nums[j] != 0]
    return list(zip(head, tail))

plan = swap_plan([0, 1, 0, 3, 12])
assert len(plan) == 2
a = [0, 1, 0, 3, 12]
for i, j in plan:
    a[i], a[j] = a[j], a[i]
assert a[-2:] == [0, 0] and swap_plan([1, 2, 0]) == []

# Bài 5
def min_swaps_binary_sorted(bits):
    z = bits.count(0)
    return sum(1 for b in bits[:z] if b == 1)

assert min_swaps_binary_sorted([1, 0, 1, 0]) == 1 and min_swaps_binary_sorted([0, 0, 1, 1]) == 0
assert min_swaps_binary_sorted([1, 1, 0, 0]) == 2
print("đáp án bài tập OK")

2 1
đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def minimumSwaps(self, nums: List[int]) -> int:
        z = nums.count(0)                       # số số 0 -> chúng phải chiếm z ô cuối
        return nums[:len(nums) - z].count(0)    # số 0 đang ở vùng đầu = số hoán đổi tối thiểu
```

Khung chung: "phân vùng bằng hoán đổi bất kỳ" -> `đáp án = số phần tử sai chỗ ở một phía`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Vùng đầu dài `n - z` (không phải `z`); mảng toàn số `0` hoặc không có số `0` cho kết quả `0`.
- [ ] Đã phân biệt hoán đổi **bất kỳ** (bài này) và hoán đổi **kề nhau**.